# **Data**

The data is  from Aquino et al (2019)

In [57]:
import subprocess
import os
import sys
from pathlib import Path

sys.path.append(os.path.abspath("../utils"))
from preprocessing_tools import *
from calculate_g2p_accuracy import calculate_accuracy

# Data pre-processing


In [58]:
current_dir = Path().resolve()

scripts_dir = current_dir.parent / "scripts"
sh_files = [f for f in os.listdir(scripts_dir) if f.endswith(".sh")]

for file in sh_files:

    file_dir = scripts_dir / file

    try:
        # Read the file in binary mode
        with open(file_dir, "rb") as f:
            content = f.read()
        
        # Replace Windows line endings (\r\n) with Linux line endings (\n)
        content = content.replace(b"\r\n", b"\n")
        
        # Write the corrected content back
        with open(file_dir, "wb") as f:
            f.write(content)
            
        print("Success! Converted line endings to LF.")
    except FileNotFoundError:
        print(f"Could not find the file at {file_dir}")

Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.


In [59]:
paths_dir = current_dir / "paths"
paths_dir.mkdir(parents=True, exist_ok=True)

getsh_files = [f for f in os.listdir(scripts_dir) if f.endswith(".sh") and f.startswith("get-")]

for file in getsh_files[:-1]:
    try:
        # 3. Run the script
        result = subprocess.run(
            ["bash", file], 
            cwd=scripts_dir,                    
            check=True,
            capture_output=True,
            text=True
        )
        print(f"You should now see {file} inside the {paths_dir} folder.")
        
    except subprocess.CalledProcessError as e:
        print(f"Script failed with exit code {e.returncode}")
        print(f"Error details:\n{e.stderr}")
    except FileNotFoundError:
        print(f"Could not find the scripts directory at: {scripts_dir}")

You should now see get-csv-test-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-csv-testref-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-csv-train-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-log-test-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-log-train-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.


In [60]:
txt_files = [f for f in os.listdir(paths_dir) if f.endswith(".txt")]

train_files = []
test_files = []
testref_files = []

for file in txt_files:
    file_path = paths_dir / file
    if "train" in file:
        train_files.append(file_path)
    elif "testref" in file:
        testref_files.append(file_path)
    else:
        test_files.append(file_path)


## **Building the train, validation and test sets**

We will begin with creating the **traininig** dictionary, and its data is sourced from `_small/train`.

In [61]:
dicts_dir = current_dir / "dicts"

create_dict(train_files[1], train_files[0], dicts_dir / "train.dict")

../data/_small/train/TGL/0812.110816.021250.csv
../data/_small/train/TGL/2359.110818.085852.csv


In [ ]:
wlist_dir = current_dir / "wlist"

create_wlist(test_files[1], wlist_dir / "test.wlist")

# **Baseline Training**

The baseline model will be from Aquino et al (2019), specifically their monolingual tagalog G2P.


In [63]:
dicts_dir = current_dir / "dicts"

create_dict(train_files[1], train_files[0], dicts_dir / "train.dict")

../data/_small/train/TGL/0812.110816.021250.csv
../data/_small/train/TGL/2359.110818.085852.csv


In [64]:
result = subprocess.run(
    ["phonetisaurus-train", "--lexicon", dicts_dir / "train.dict", "--seq2_del"],
    capture_output=True,
    text=True
)

print(result.stderr)

INFO:phonetisaurus-train:2026-03-30 07:41:21:  Checking command configuration...
INFO:phonetisaurus-train:2026-03-30 07:41:21:  Checking lexicon for reserved characters: '}', '|', '_'...
INFO:phonetisaurus-train:2026-03-30 07:41:21:  Aligning lexicon...
INFO:phonetisaurus-train:2026-03-30 07:41:23:  Training joint ngram model...
INFO:phonetisaurus-train:2026-03-30 07:41:23:  Converting ARPA format joint n-gram model to WFST format...
INFO:phonetisaurus-train:2026-03-30 07:41:23:  G2P training succeeded: train/model.fst



In [65]:
model_dir = current_dir / "train"

output_txt = current_dir / "output" / "baseline_output.txt"

with open(output_txt, "w") as f:
    subprocess.run(
        [
            "phonetisaurus-apply",
            "--model", str(model_dir / "model.fst"),
            "--word_list", str(wlist_dir / "test.wlist"),
        ],
        stdout=f,
        text=True
    )



In [66]:
create_trxn(test_files[1], output_txt)

calculate_accuracy(test_files[0], testref_files[0])

../data/_small/test/TGL/0156.110816.035620.csv
../data/_small/test/TGL/0156.110816.035620.csv
../data/_small/testref/TGL/0156.110816.035620.csv
total phones in reference: 11089
total edit distance:       651
phone error rate:          0.05870682658490396
